In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
from scipy.stats import kruskal
import re

warnings.filterwarnings('ignore')

#plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

In [ ]:
final_review_categories = pd.read_csv("../../../../data/preprocessed/tableau_issue_long.csv")

In [ ]:
# =========================================================
# 01. Tableau용 CSV 저장 전용 설정
# =========================================================

import pandas as pd
import csv

# 원본 보호
tableau_issue = final_review_categories.copy()

print("원본 크기:", tableau_issue.shape)
display(tableau_issue.head())

원본 크기: (121354, 8)


,recommendationid,appid,primary_genre,sentiment,review,clean_review,final_category,llm_reason
0,18699465,324470,Racing,positive,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",bgm good graphics good but slip effect is too ...,난이도/밸런스,NaN
1,18699465,324470,Racing,positive,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",bgm good graphics good but slip effect is too ...,아트/비주얼,NaN
2,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! \n\...",this game is like a zen garden i love it pros ...,업데이트/개발,NaN
3,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! \n\...",this game is like a zen garden i love it pros ...,조작/UX,NaN
4,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! \n\...",this game is like a zen garden i love it pros ...,음악/사운드,NaN


In [ ]:
tableau_issue['appid'].nunique()

NameError: name 'tableau_issue' is not defined

In [ ]:
# =========================================================
# 02. Tableau에서 CSV 깨짐을 만드는 문자 정리
# =========================================================

# Tableau로 보낼 때 문자열로 관리할 컬럼들
text_cols = [
    "primary_genre",
    "sentiment",
    "review",
    "clean_review",
    "final_category",
    "llm_reason"
]

for col in text_cols:
    if col in tableau_issue.columns:
        tableau_issue[col] = (
            tableau_issue[col]
            .astype("string")
            .fillna("")
            .str.replace("\r", " ", regex=False)
            .str.replace("\n", " ", regex=False)
            .str.replace("\t", " ", regex=False)
            .str.replace('"', "'", regex=False)
            .str.strip()
        )

display(tableau_issue.head())

,recommendationid,appid,primary_genre,sentiment,review,clean_review,final_category,llm_reason
0,18699465,324470,Racing,positive,"BGM - GOOD GRAPHICS - GOOD But, Slip effect ...",bgm good graphics good but slip effect is too ...,난이도/밸런스,
1,18699465,324470,Racing,positive,"BGM - GOOD GRAPHICS - GOOD But, Slip effect ...",bgm good graphics good but slip effect is too ...,아트/비주얼,
2,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,업데이트/개발,
3,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,조작/UX,
4,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,음악/사운드,


In [ ]:
# =========================================================
# 03. ID 컬럼 정리
# =========================================================

id_cols = ["recommendationid", "appid"]

for col in id_cols:
    if col in tableau_issue.columns:
        tableau_issue[col] = (
            tableau_issue[col]
            .astype("string")
            .str.replace(".0", "", regex=False)
            .str.strip()
        )

print(tableau_issue.dtypes)
display(tableau_issue.head())

recommendationid    string
appid               string
primary_genre       string
sentiment           string
review              string
clean_review        string
final_category      string
llm_reason          string
dtype: object


,recommendationid,appid,primary_genre,sentiment,review,clean_review,final_category,llm_reason
0,18699465,324470,Racing,positive,"BGM - GOOD GRAPHICS - GOOD But, Slip effect ...",bgm good graphics good but slip effect is too ...,난이도/밸런스,
1,18699465,324470,Racing,positive,"BGM - GOOD GRAPHICS - GOOD But, Slip effect ...",bgm good graphics good but slip effect is too ...,아트/비주얼,
2,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,업데이트/개발,
3,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,조작/UX,
4,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,음악/사운드,


In [ ]:
# =========================================================
# 04. LLM 에러 메시지가 섞인 행 제거
# =========================================================

error_patterns = [
    "Quota exceeded",
    "429",
    "ai.google.dev",
    "generativelanguage",
    "RetryInfo",
    "PERMISSION_DENIED",
    "RESOURCE_EXHAUSTED"
]

before_rows = len(tableau_issue)

mask_error = pd.Series(False, index=tableau_issue.index)

for col in ["review", "clean_review", "llm_reason"]:
    if col in tableau_issue.columns:
        for pattern in error_patterns:
            mask_error = mask_error | tableau_issue[col].str.contains(
                pattern,
                case=False,
                na=False,
                regex=False
            )

tableau_issue_clean = tableau_issue[~mask_error].copy()

after_rows = len(tableau_issue_clean)

print("제거 전 행 수:", before_rows)
print("제거 후 행 수:", after_rows)
print("제거된 행 수:", before_rows - after_rows)

display(tableau_issue_clean.head())

제거 전 행 수: 121354
제거 후 행 수: 99677
제거된 행 수: 21677


,recommendationid,appid,primary_genre,sentiment,review,clean_review,final_category,llm_reason
0,18699465,324470,Racing,positive,"BGM - GOOD GRAPHICS - GOOD But, Slip effect ...",bgm good graphics good but slip effect is too ...,난이도/밸런스,
1,18699465,324470,Racing,positive,"BGM - GOOD GRAPHICS - GOOD But, Slip effect ...",bgm good graphics good but slip effect is too ...,아트/비주얼,
2,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,업데이트/개발,
3,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,조작/UX,
4,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! p...",this game is like a zen garden i love it pros ...,음악/사운드,


In [ ]:
# =========================================================
# 05. Tableau용 CSV 안전 저장
# =========================================================

SAVE_PATH = "../../../../data/preprocessed/tableau_issue_long_clean.csv"

tableau_issue_clean.to_csv(
    SAVE_PATH,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n"
)

print("Tableau용 CSV 저장 완료:", SAVE_PATH)

Tableau용 CSV 저장 완료: ../../../../data/preprocessed/tableau_issue_long_clean.csv
